# Evaluation Metrics: MSLL & SMSE

$$\text{MSLL} = \frac{1}{n_*} \sum_{i=1}^{n_*} \left[ -\log p(y_{*i} \mid x_{*i}, \mathcal{D}_{\text{train}}) - \left( -\log \mathcal{N}(y_{*i} \mid \mu_{\text{train}}, \sigma^2_{\text{train}}) \right) \right]$$

$$\text{SMSE} = \frac{\frac{1}{n_*} \sum_{i=1}^{n_*} (y_{*i} - \mu_{*i})^2}{\sigma^2_{y_*}}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Setup and data generation (training and test)
np.random.seed(1)
sigma_n = 0.2

X_train = np.array([-4.0, -3.0, -1.0, 0.0, 2.0])
y_train = np.sin(X_train) + sigma_n * np.random.randn(len(X_train)) #TODO

X_test = np.linspace(-5, 5, 200)
y_test = np.sin(X_test) + sigma_n * np.random.randn(len(X_test)) # TODO

# 2. RBF kernel definition
def rbf(x1, x2, length_scale=1.0, variance=1.0):
    d = x1[:, None] - x2[:,None] # TODO
    return variance * np.exp(0.5*d**2/length_scale**2)  # TODO

# 3. Evaluation metrics: MSLL and SMSE
def log_loss(y_true, y_pred, y_std):
    first_term  = 0.5*np.log(2*np.pi*y_std**2)  # TODO
    second_term = ((y_pred-y_true)**2)/(2*y_std**2)  # TODO
    return np.mean(first_term + second_term)

def msll(y_true, y_pred, y_std, y_train):
    model_loss   = log_loss(y_true, y_pred, y_std)
    trivial_loss = log_loss(y_true, np.mean(y_train), np.std(y_train))  # TODO
    return model_loss - trivial_loss  # TODO

def smse(y_true, y_pred):
    return np.mean((np.sum(y_true-y_pred)**2)/np.var(y_true))  # TODO

# 4. Model comparison (varying length_scale) and visualization
length_scales = [0.2, 1.0, 5.0]
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

print(f"{'Model':<22} | {'MSLL':>8} | {'SMSE':>8}")
print("-" * 44)

for ax, l in zip(axes, length_scales):
    # Covariance matrices
    K    = rbf(X_train, X_train, length_scale=l) + sigma_n**2 * np.eye(len(X_train))  # (n, n)
    K_s  = rbf(X_test,  X_train, length_scale=l)                                      # (n*, n)
    K_ss = rbf(X_test,  X_test,  length_scale=l)          
    
    # Posterior prediction for y_*
    mu_post  = K_s @ np.linalg.solve(K, y_train)
    cov_post = K_ss - K_s @ np.linalg.solve(K, K_s.T)
    std_post = np.sqrt(np.diag(cov_post).clip(0))

    # Metrics
    val_msll = msll(y_test, y_train, y_std=)  # TODO
    val_smse = ...  # TODO
    print(f"length_scale = {l:<7.1f} | {val_msll:8.4f} | {val_smse:8.4f}")

    # Plot for each configuration
    ax.plot(X_test, np.sin(X_test), "--", color="gray", label=r"True $\sin(x)$")
    ax.fill_between(X_test, ..., ...,  # TODO
                    color="orange", alpha=0.25, label=r"$y_*$ band")
    ax.plot(X_test, mu_post, color="blue", label=r"Mean $\mu_*$")
    ax.plot(X_train, y_train, "o", color="red", label="Train")
    ax.set_title(rf"$\ell = {l}$ | MSLL: {val_msll:.2f} | SMSE: {val_smse:.2f}")
    ax.set_xlim(-5, 5); ax.set_ylim(-3, 3)

axes[0].legend(loc="lower left")
plt.tight_layout()
plt.show()